# DORA Metrics for Deployment Frequency Only

## Objective

### What do we want to be able to show for deployment frequency?

- Total deployment frequency over time.
  - Regardless of appID, Env, or Status
- Then split by separately:
  - AppID
  - Env
  - Status
- Then maybe a matrix of these options
 - AppID / Env / Status

## Environment

In [ ]:
print("Kernel is working.")

In [ ]:
from dash import dcc
import pandas as pd
import matplotlib.pyplot as plt
import plotly.express as px

## Data Ingestion

**Output:** pandas dataframe *df_deployments*.

In [ ]:
# Useful reference: <https://pandas.pydata.org/docs/getting_started/intro_tutorials/09_timeseries.html>
raw_file_path = '/home/lnx_workspaces/bpp_projects/bpp_module2_project/doraview/data/json/deployment_frequency.json'

# Try reading with explicit encoding and error handling
try:
    df_deployments = pd.read_json(raw_file_path, encoding='utf-8', convert_dates=["deployed_at"])
    print("\nDataframe info:")
    print(df_deployments.info())
    print("\nFirst 5 rows:")
    print(df_deployments.head())
except Exception as e:
    print(f"Error: {str(e)}")

## Data Manipulation

Manipulate *df_deployments* into format needed to produce figures.

- New columns to use for grouping:
  - row quater
  - row month

### Add aggregated datetime columns

In [ ]:
# Add new row and quater columns
df_deployments["month"] = df_deployments["deployed_at"].dt.month
df_deployments["qtr"] = df_deployments["deployed_at"].dt.quarter

df_deployments.head(100)

In [ ]:
# Reduce number of columns and check data types carry over.
df_deploy_reduced = df_deployments[["application_id","application_name","environment","status","deployed_at","month","qtr"]]
df_deploy_reduced.info()

### New df with percent failure / success of deployment

### Example how to

Sourced from <https://sparkbyexamples.com/pandas/pandas-percentage-total-with-groupby/>

In [ ]:
# Create a Pandas DataFrame
import pandas as pd
import numpy as np
technologies= {
    'Courses':["Spark","PySpark","Spark","Python","PySpark"],
    'Fee' :[22000,25000,23000,24000,26000],
    'Duration':['30days','50days','30days', None,np.nan]
          }
df = pd.DataFrame(technologies)
print(df)

In [ ]:
# Using DataFrame.agg() Method
df2 = df.groupby(['Courses', 'Fee']).agg({'Fee': 'sum'})
print(df2)

In [ ]:
# Percentage by lambda and DataFrame.apply() method
df3 = df2.groupby(level=0).apply(lambda x:100 * x / float(x.sum()))
print(df3)

In [ ]:
# Using DataFrame.div() method.
df2 = df.groupby(['Courses', 'Fee']).agg({'Fee': 'sum'})
Courses = df.groupby(['Courses']).agg({'Fee': 'sum'})
df2.div(Courses, level='Courses') * 100
print(df2)

In [ ]:
# Using DataFrame.transform() method
df['%'] = 100 * df['Fee'] / df.groupby('Courses')['Fee'].transform('sum')
print(df)

In [ ]:
# Alternative method of DataFrame.transform() by lambda functions
df['Courses_Fee'] = df.groupby(['Courses'])['Fee'].transform(lambda x: x/x.sum())
print(df)

In [ ]:
# Caluclate groupby with DataFrame.rename() and DataFrame.transform() with lambda functions
df2=df.groupby(['Courses', 'Fee'])['Fee'].sum().rename("Courses_fee").groupby(level = 0).transform(lambda x: x/x.sum())
print(df2)

## Figures

### ChatGPT - Deployment Frequency (DF)

**Best Graph Type:** Bar Chart (vertical) or Line Chart — plotted over time

**Why:** Deployment Frequency is a count of deployments per day/week/month.

**You want to show:**

- How deployment cadence changes over time
- Peaks, slow periods, or stability
- Differences between environments (if applicable)

Bar charts make discrete counts very clear.
Line charts work well if you want to show smoothed trends across longer periods.

**Optional secondary views:**

- Heatmap Calendar (GitHub-style commit calendar)
- Grouped bars (different microservices or applications per time window)

### Styling

['#636EFA', # the plotly blue you can see above
 '#EF553B',
 '#00CC96',
 '#AB63FA',
 '#FFA15A',
 '#19D3F3',
 '#FF6692',
 '#B6E880',
 '#FF97FF',
 '#FECB52']

### Multi-Group Display

In [ ]:
df_deploy_multi_group = df_deployments.groupby([
	"application_id",
	"application_name",
	"month",
	"environment",
	"status"
	])["month"].count().reset_index(name="count")

df_deploy_multi_group.head(100)

In [ ]:
# display dataframe as figure
fig_month_multi_bar = px.bar(
	data_frame=df_deploy_multi_group,
	title="Total Monthly Deployments",	# Label for the figure.
	x="month",							# Column for use on x-axis
	y="count",							# Column for use on y-axis
	subtitle="All Applications",
)

fig_month_multi_bar.update_yaxes(
	title_text="Deployment Count"		# Update title for y-axis
)

fig_month_multi_bar.update_xaxes(
	title_text="Deployment Month",		# Update title for x-axis
	tickvals=list(range(1,13)),			# Specify values of x-axis
	ticktext=['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun', 'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec']	# Allocate text to tick values.
)

fig_month_multi_bar.show()

### Monthly Deployments

In [ ]:
# create grouped dataframe
df_deploy_group_month = df_deployments.groupby(["month"])["month"].count().reset_index(name="count")

df_deploy_group_month.head(100)

In [ ]:
# display dataframe as figure
fig_month_all_bar = px.bar(
	data_frame=df_deploy_group_month,
	title="Total Monthly Deployments",	# Label for the figure.
	x="month",							# Column for use on x-axis
	y="count",							# Column for use on y-axis
	subtitle="All Applications",
)

fig_month_all_bar.update_yaxes(
	title_text="Deployment Count"		# Update title for y-axis
)

fig_month_all_bar.update_xaxes(
	title_text="Deployment Month",		# Update title for x-axis
	tickvals=list(range(1,13)),			# Specify values of x-axis
	ticktext=['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun', 'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec']	# Allocate text to tick values.
)

fig_month_all_bar.show()

In [ ]:
# display dataframe as figure
fig_month_all_line = px.line(
	data_frame=df_deploy_group_month,
	title="Total Monthly Deployments",	# Label for the figure.
	x="month",							# Column for use on x-axis
	y="count",							# Column for use on y-axis
	subtitle="All Applications"
)

fig_month_all_line.update_yaxes(
	title_text="Deployment Count"		# Update title for y-axis
)

fig_month_all_line.update_xaxes(
	title_text="Deployment Month",		# Update title for x-axis
	tickvals=list(range(1,13)),			# Specify values of x-axis
	ticktext=['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun', 'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec']	# Allocate text to tick values.
)

fig_month_all_line.show()

### Monthly Deployments by Application

In [ ]:
# create grouped dataframe
df_deploy_group_month_app = df_deployments.groupby(["application_name","month"])["month"].count().reset_index(name="count")

df_deploy_group_month_app.head(100)

In [ ]:
# display dataframe as figure
fig_month_all_bar = px.bar(
	data_frame=df_deploy_group_month_app,
	title="Total Monthly Deployments by Application",
	x="month",
	y="count",
	color="application_name",	# Changing this determines the divisions
)

fig_month_all_bar.update_yaxes(
	title_text="Deployment Count"
)

fig_month_all_bar.update_xaxes(
	title_text="Deployment Month",
	tickvals=list(range(1,13)),
	ticktext=['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun', 'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec'],
)

fig_month_all_bar.show()

In [ ]:
# display dataframe as figure
fig_month_all_line = px.line(
	data_frame=df_deploy_group_month_app,
	title="Total Monthly Deployments by Application",
	x="month",
	y="count",
	color="application_name"
)

fig_month_all_line.update_yaxes(
	title_text="Deployment Count"
)

fig_month_all_line.update_xaxes(
	title_text="Deployment Month",
	tickvals=list(range(1,13)),
	ticktext=['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun', 'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec'],
)

fig_month_all_line.show()

### Monthly Deployments by Environment

In [ ]:
# create grouped dataframe
df_deploy_group_month_env = df_deployments.groupby(["environment","month"])["month"].count().reset_index(name="count")

df_deploy_group_month_env.head(100)

In [ ]:
# display dataframe as figure
fig_month_env_bar = px.bar(
	data_frame=df_deploy_group_month_env,
	title="Total Deployments by Environment",
	x="month",
	y="count",
	color="environment",
	color_discrete_map={"staging":"#636EFA", "production":"#EF553B"},
)

fig_month_env_bar.update_yaxes(
	title_text="Deployment Count"
)

fig_month_env_bar.update_xaxes(
	title_text="Deployment Month",
	tickvals=list(range(1,13)),
	ticktext=['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun', 'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec'],
)

fig_month_env_bar.show()

In [ ]:
# display dataframe as figure
fig_month_env_line = px.line(
	data_frame= df_deploy_group_month_env,
	title="Total Deployments by Environment",
	x="month",
	y="count",
	color="environment",
	color_discrete_map={
		"staging":"#636EFA",
		"production":"#EF553B"
		},
)

fig_month_env_line.update_yaxes(
	title_text="Deployment Count"
)

fig_month_env_line.update_xaxes(
	title_text="Deployment Month",
	tickvals=list(range(1,13)),
	ticktext=['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun', 'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec'],
)

fig_month_env_line.show()

### Monthly Deployments by Status

In [ ]:
# create grouped dataframe
df_deploy_group_month_status = df_deployments.groupby(["status","month"])["month"].count().reset_index(name="count")

df_deploy_group_month_status.head(100)

In [ ]:
# display dataframe as figure
fig_month_stat_bar = px.bar(
	df_deploy_group_month_status,
	title="Total Deployments by Status",
	x="month",
	y="count",
	color="status",
	color_discrete_map={
		"success":"#636EFA",
		"failed":"#EF553B"
		},
)

fig_month_stat_bar.update_yaxes(
	title_text="Deployment Count"
)

fig_month_stat_bar.update_xaxes(
	title_text="Deployment Month",
	tickvals=list(range(1,13)),
	ticktext=['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun', 'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec'],
)

fig_month_stat_bar.show()

In [ ]:
# display dataframe as figure
fig_month_stat_line = px.line(
	df_deploy_group_month_status,
	title="Total Deployments by Status",
	x="month",
	y="count",
	color="status",
	color_discrete_map={
		"success":"#636EFA",
		"failed":"#EF553B"
		},
)

fig_month_stat_line.update_yaxes(
	title_text="Deployment Count"
)

fig_month_stat_line.update_xaxes(
	title_text="Deployment Month",
	tickvals=list(range(1,13)),
	ticktext=['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun', 'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec'],
)

fig_month_stat_line.show()